# Text To Speech (TTS)

**Module:** 16 — Speech AI

Synthesis quality, SSML/prosody, streaming audio, and voice cloning ethics.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain TTS pipelines: linguistic analysis → acoustic model → vocoder
- Control prosody with SSML-like markup
- List quality dimensions (naturalness, similarity, latency)
- Apply ethical controls for cloning and disclosure


## TTS Concepts

### Definition
TTS converts text (or SSML) into audible speech waveforms.

### Why it matters
Voice agents die without clear, low-latency, on-brand speech.

### How it works
Normalize text → phonemes/linguistic feats → mel/codec tokens → neural vocoder → audio stream.

### Intuition
A musician reading a score, then a synthesizer rendering timbre.

### Pitfalls
- Sending raw LLM text with markdown/tables to TTS
- No streaming → long silence before speech

### When to use
IVR, agents, accessibility readers, media narration.


### Quality dimensions

| Dimension | Means | Trade |
|-----------|-------|-------|
| Naturalness | Human-like | Compute |
| Similarity | Matches a voice | Ethics/consent |
| Intelligibility | Understood in noise | May reduce style |
| Latency | Time-to-first-audio | Chunk size |
| Consistency | Same name/pronunciation | Pron lexicons |

```mermaid
flowchart LR
  T[Text] --> N[Normalize]
  N --> A[Acoustic / codec model]
  A --> V[Vocoder]
  V --> O[PCM / Opus stream]
```


In [ ]:
# Demo 1: TTS request shapes (OpenAI-style placeholder)
import os, json
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
req = {
    "model": "gpt-4o-mini-tts",
    "voice": "alloy",
    "input": "Your order 4455 has been canceled.",
    "format": "wav",
}
print(json.dumps(req, indent=2))
print("Authorization: Bearer", OPENAI_API_KEY[:12] + "...")


In [ ]:
# Demo 2: sanitize LLM text before TTS
import re
def tts_sanitize(text: str) -> str:
    text = re.sub(r"[`*_#]+", " ", text)
    text = re.sub(r"\bhttps?://\S+", " link ", text)
    text = text.replace("|", " ")
    text = re.sub(r"\s+", " ", text).strip()
    # expand a few abbreviations
    text = text.replace("approx.", "approximately")
    return text
print(tts_sanitize("**Done!** See https://example.com/x | approx. 3 mins"))


## SSML & Prosody

### Definition
SSML (and vendor markup) controls breaks, rate, pitch, emphasis, and say-as interpretations.

### Why it matters
Phone numbers, currencies, and confirmations need predictable pronunciation.

### How it works
Wrap critical spans with say-as; insert breaks after confirmations; keep rate moderate for IVR.

### Intuition
Punctuation is stage direction for the voice.

### Pitfalls
- Overusing pitch tricks (carnival bot)
- Not testing locale-specific say-as

### When to use
IVR, banking confirmations, multilingual prompts.


In [ ]:
# Demo 3: minimal SSML builder
def ssml_confirm(order_id: str, amount: str) -> str:
    return f'''<speak>
  Your order <say-as interpret-as="digits">{order_id}</say-as>
  for <say-as interpret-as="currency">{amount}</say-as>
  is canceled.
  <break time="300ms"/>
  Is there anything else?
</speak>'''
print(ssml_confirm("4455", "$19.99"))


In [ ]:
# Demo 4: chunk text for streaming TTS
def chunk_for_tts(text, max_chars=80):
    parts, buf = [], []
    for word in text.split():
        buf.append(word)
        if sum(len(w)+1 for w in buf) >= max_chars and word.endswith((".", "?", "!")):
            parts.append(" ".join(buf)); buf=[]
    if buf: parts.append(" ".join(buf))
    return parts
print(chunk_for_tts("Your order is canceled. Thanks for calling Acme Support. How else can I help?"))


## Voice Cloning Ethics

### Definition
Cloning reproduces a target speaker's voice from samples — powerful and abuse-prone.

### Why it matters
Fraud, non-consensual deepfakes, and brand impersonation are real threats.

### How it works
Require explicit consent, watermarking when available, allow-lists, human review for high-risk, disclose synthetic speech where required.

### Intuition
A voice is an identity factor — treat it like a biometric.

### Pitfalls
- Scraping celebrity audio to clone
- No user disclosure in sensitive contexts

### When to use
Personalized assistants with consent; never for impersonation.


In [ ]:
# Demo 5: clone policy gate
def allow_clone(request: dict) -> str:
    if not request.get("signed_consent"):
        return "deny: consent"
    if request.get("subject_type") == "celebrity" and not request.get("legal_ok"):
        return "deny: legal"
    if request.get("use_case") in {"fraud", "impersonate_bank"}:
        return "deny: use_case"
    return "allow_with_watermark"
print(allow_clone({"signed_consent": True, "subject_type": "employee", "use_case": "personal_assistant"}))
print(allow_clone({"signed_consent": False, "use_case": "personal_assistant"}))
print(allow_clone({"signed_consent": True, "use_case": "impersonate_bank"}))


In [ ]:
# Demo 6: pronunciation lexicon
LEX = {"acme": "ACK-mee", "qemu": "KEE-myoo", "sql": "S-Q-L"}
def apply_lexicon(text: str) -> str:
    out = []
    for w in text.split():
        key = w.lower().strip(".,!?")
        out.append(LEX.get(key, w))
    return " ".join(out)
print(apply_lexicon("Welcome to Acme cloud SQL support"))


### When to pick which TTS

| Need | Preference |
|------|------------|
| Lowest latency agent | Streaming neural TTS |
| Brand custom voice | Cloned/designed voice + consent |
| High volume IVR | Cost-efficient neural / hybrid |
| Accessibility screen reader | Clear, adjustable rate |


### Checklist — TTS production

- [ ] Sanitizer strips markdown/URLs
- [ ] Pronunciation lexicon for brand terms
- [ ] Time-to-first-audio measured
- [ ] Clone consent + watermark path
- [ ] Locale/say-as tests for numbers


### Try it yourself — TTS & SSML

1. Add telephone say-as for a support number.
2. Split LLM answers into speakable sentences.
3. Design a disclosure string for synthetic voice.

**Stretch:** Generate audio with a live TTS API using env keys only.


### Try it yourself — Ethics

1. Write an accept/deny matrix for clone requests.
2. List 5 fraud scenarios involving cloned voices.


## Knowledge Check

**Q1.** Why sanitize markdown before TTS?

<details><summary>Answer</summary>

Models often emit **, links, and tables that sound awful or leak URLs when spoken.

</details>

**Q2.** What is time-to-first-audio?

<details><summary>Answer</summary>

Latency until the caller hears the first sound — critical for perceived responsiveness.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `vocoder` | Renders acoustic features to waveform |
| `SSML` | Speech synthesis markup language |
| `prosody` | Rhythm, stress, intonation |
| `MOS` | Listening quality score |
| `voice clone` | Synthesize a target speaker voice |
| `watermark` | Inaudible/traceable synthetic marker |


## Key Takeaways

- TTS quality = naturalness + intelligibility + latency
- SSML/lexicons make numbers and brands reliable
- Stream audio; sanitize LLM text
- Cloning needs consent, policy, and disclosure


## Production Incident Patterns — TTS

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "TTS",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — TTS

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("TTS", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — TTS ops

1. Draft an on-call runbook bullet list for TTS when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
